# Beyond - Reinforcement Learning for LLMs

This notebook runs a deliberately small group-relative REINFORCE loop on LoRA adapters over BaseLM: use format learning as a smoke test, measure arithmetic on held-out prompts, then audit a deliberately sloppy verifier. Every required experiment starts from the same untouched BaseLM and fresh zero-delta adapters. Requires the BaseLM artifact (`./baselm.sh`) and the Module 13B LoRA implementation.

1. Read the lesson page (`docs/beyond/rl.md`).
2. Open this notebook with `./notebook.sh rl`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import gc
import random

import torch
import matplotlib.pyplot as plt

from g2c.artifacts.baselm import (
    baselm_artifact_exists, load_huggingface_model_artifact,
)
from g2c.lora import (
    LoRAModel, inject_lora, mark_only_lora_trainable,
)
from g2c.rl import (
    GRPOTrainer, arithmetic_choice_task, format_task, sample_group,
    verify_arithmetic, verify_arithmetic_sloppy, verify_format,
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
HAVE_BASELM = baselm_artifact_exists()
if not HAVE_BASELM:
    print("BaseLM artifact not found — run ./baselm.sh, then restart this notebook.")
else:
    print(f"BaseLM found; device: {device}")

In [ ]:
LORA_RANK = 8
LORA_ALPHA = 8.0
LORA_TARGETS = {"q_proj", "v_proj"}


def fresh_lora_pair():
    """Fresh zero-delta LoRA policy plus an untouched frozen reference."""
    policy_art = load_huggingface_model_artifact(device=device)
    policy = policy_art.model
    inject_lora(
        policy, LORA_TARGETS, rank=LORA_RANK, alpha=LORA_ALPHA
    )
    trainable, total = mark_only_lora_trainable(policy)

    ref = load_huggingface_model_artifact(device=device).model
    for parameter in ref.parameters():
        parameter.requires_grad_(False)
    return LoRAModel(policy), ref, policy_art.tokenizer, trainable, total


if HAVE_BASELM:
    print("Each experiment will load a fresh LoRA policy/reference pair.")

## Exercise 1 — Build the LoRA policy and a held-out baseline

Inject rank-8 LoRA adapters into BaseLM's query and value projections, freeze the base weights, and print the trainable fraction. Keep training and evaluation prompts disjoint. The policy starts exactly at the frozen reference because every adapter delta is zero. The format task supplies the opening `{"answer":` prefix in the prompt, making valid and invalid completions discoverable without weakening the binary verifier.

In [ ]:
if HAVE_BASELM:
    format_model, format_ref, tokenizer, trainable, total = fresh_lora_pair()
    print(f"LoRA parameters: {trainable:,} / {total:,} "
          f"({100 * trainable / total:.3f}%)")

    format_train = [format_task(random.Random(1_000 + i)) for i in range(24)]
    format_eval = [format_task(random.Random(2_000 + i)) for i in range(24)]

    common_cfg = dict(
        group_size=6, lr=2e-4, kl_coef=0.05,
        temperature=0.8, eos_id=tokenizer.eos_token_id, seed=0,
    )
    format_cfg = {**common_cfg, "max_new_tokens": 12}
    format_trainer = GRPOTrainer(
        format_model, format_ref, tokenizer, format_train, verify_format,
        **format_cfg,
    )
    format_baseline = format_trainer.evaluate(format_eval)
    print(f"held-out pre-RL format pass rate: {format_baseline:.1%}")

## Exercise 2 — RL on format

Run only ten updates against `verify_format` (complete a JSON object with an `"answer"` key). This is a pipeline smoke test, not the module's main result. Watch reward and `skipped`: a group whose completions all receive the same score carries no relative signal.

In [ ]:
if HAVE_BASELM:
    format_history = format_trainer.train(10, log_every=5)

    plt.figure(figsize=(7, 4))
    plt.plot(format_history["mean_reward"], label="mean group reward")
    plt.plot(format_history["skipped"], alpha=0.4,
             label="skipped (degenerate group)")
    plt.xlabel("step"); plt.legend(); plt.title("GRPO on the format task")
    plt.show()

    format_post = format_trainer.evaluate(format_eval)
    print(f"held-out format pass rate: "
          f"{format_baseline:.1%} → {format_post:.1%}")

In [ ]:
if HAVE_BASELM:
    group = sample_group(format_model, tokenizer, format_eval[0]["prompt"], 4,
                         max_new_tokens=12, temperature=0.8,
                         eos_id=tokenizer.eos_token_id)
    print(f"prompt: {format_eval[0]['prompt']}")
    for text in group.texts:
        print(
            f"  → {text!r}  (reward {verify_format(format_eval[0], text)})"
        )

In [ ]:
"Question: Report the held-out format pass rate before and after the smoke run. What did reward and the skipped fraction do? Explain why a skipped group can mean either all failures or all successes, so the metric must be interpreted alongside held-out generations."
"Answer: "

## Exercise 3 — Main run: arithmetic

Release the format models and start again from untouched BaseLM plus fresh zero-delta adapters. Each two-digit addition prompt supplies two numeric options; calibration showed that free response made almost every small group all-wrong, while the choice task preserves an arithmetic decision and supplies usable contrast. Train on one prompt set and evaluate greedy generated answers on a disjoint set. Held-out pass rate is the result; training reward, KL, sampled entropy, skip rate, and sample text are diagnostics. Do not assume improvement—the run must earn that claim.

In [ ]:
if HAVE_BASELM:
    del format_trainer, format_model, format_ref
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()

    arith_train = [
        arithmetic_choice_task(random.Random(3_000 + i)) for i in range(48)
    ]
    arith_eval = [
        arithmetic_choice_task(random.Random(4_000 + i)) for i in range(48)
    ]
    arith_model, arith_ref, tokenizer, _, _ = fresh_lora_pair()
    arith_cfg = {**common_cfg, "max_new_tokens": 8}
    arith_trainer = GRPOTrainer(
        arith_model, arith_ref, tokenizer, arith_train,
        verify_arithmetic, **arith_cfg,
    )
    arith_baseline = arith_trainer.evaluate(arith_eval)
    arith_history = arith_trainer.train(40, log_every=10)
    arith_post = arith_trainer.evaluate(arith_eval)
    print(f"held-out arithmetic pass rate: "
          f"{arith_baseline:.1%} → {arith_post:.1%}")

    fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
    for ax, key, title in zip(
        axes.flat,
        ["mean_reward", "kl", "sample_entropy", "skipped"],
        ["training reward", "KL estimate",
         "sampled-token entropy estimate", "skipped group"],
    ):
        ax.plot(arith_history[key]); ax.set_title(title)
    fig.supxlabel("step"); fig.tight_layout(); plt.show()

    group = sample_group(
        arith_model, tokenizer, arith_eval[0]["prompt"], 4,
        max_new_tokens=8, temperature=0.8,
        eos_id=tokenizer.eos_token_id,
    )
    print(f"prompt: {arith_eval[0]['prompt']}")
    for text in group.texts:
        print(f"  → {text!r}  reward={verify_arithmetic(arith_eval[0], text)}")

In [ ]:
"Question: Report the held-out arithmetic pass rate before and after RL, then use the reward, KL, sampled-entropy, skip curves, and generated answers to qualify the result. Which evidence supports learning, drift, or no clear change? Why is training reward alone insufficient?"
"Answer: "

## Exercise 4 — Audit a sloppy reward

Release the honest arithmetic policy and start from the same BaseLM/LoRA initialization again, changing only the verifier. `verify_arithmetic_sloppy` rewards an answer appearing anywhere, not the model's final answer. Compare its held-out score against the honest verifier and inspect generations. The exploit is an empirical outcome: document what your run did, including if the short run failed to discover the gap. **In RL, the reward curve is a claim; independent evaluation and samples are the evidence.**

In [ ]:
if HAVE_BASELM:
    del arith_trainer, arith_model, arith_ref
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()

    sloppy_model, sloppy_ref, tokenizer, _, _ = fresh_lora_pair()
    sloppy = GRPOTrainer(
        sloppy_model, sloppy_ref, tokenizer, arith_train,
        verify_arithmetic_sloppy, **arith_cfg,
    )
    sloppy_claimed_before = sloppy.evaluate(
        arith_eval, verifier=verify_arithmetic_sloppy
    )
    sloppy_honest_before = sloppy.evaluate(
        arith_eval, verifier=verify_arithmetic
    )
    sloppy_history = sloppy.train(40, log_every=10)
    sloppy_claimed_after = sloppy.evaluate(
        arith_eval, verifier=verify_arithmetic_sloppy
    )
    sloppy_honest_after = sloppy.evaluate(
        arith_eval, verifier=verify_arithmetic
    )
    print(
        "held-out claimed pass rate: "
        f"{sloppy_claimed_before:.1%} → {sloppy_claimed_after:.1%}"
    )
    print(
        "held-out honest pass rate:  "
        f"{sloppy_honest_before:.1%} → {sloppy_honest_after:.1%}"
    )
    group = sample_group(
        sloppy_model, tokenizer, arith_eval[0]["prompt"], 4,
                         max_new_tokens=8, temperature=0.8,
                         eos_id=tokenizer.eos_token_id)
    print("samples under the SLOPPY reward — read them, don't trust the curve:")
    for text in group.texts:
        honest = verify_arithmetic(arith_eval[0], text)
        claimed = verify_arithmetic_sloppy(arith_eval[0], text)
        print(f"  claimed {claimed}, honest {honest}: {text!r}")

In [ ]:
"Question: Report the sloppy verifier's held-out claimed and honest pass rates, then cite your generated samples. What did the verifier reward, what behavior did this run find (or fail to find), and what smallest verifier change closes the gap? Do not infer an exploit from training reward alone."
"Answer: "

## Exercise 5 — Remove the KL leash (optional)

Start a fresh arithmetic run with `kl_coef=0.0`. Compare held-out pass rate, KL, sampled entropy, and generated text against Exercise 3. A short run may not visibly collapse; report what happened rather than claiming the leash is unnecessary or that collapse was guaranteed.

## Exercise 6 — Group-size sweep (optional)

Re-run the format smoke test at `group_size` in {2, 8, 16} with a fixed total rollout budget. Small groups estimate relative performance from fewer attempts; large groups spend more rollouts on each prompt. Compare reward variance and the empirical degenerate-group fraction without assuming where the knee must be.

## A note on memory

Every required run uses LoRA. Only adapter parameters receive gradients and AdamW state; the frozen BaseLM policy weights and frozen reference still occupy memory and both still run forward passes. The notebook releases each pair before loading the next so experiments remain independent without accumulating model copies.

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.